# Competencia ML1: ejemplo de entrenamiento

Pipeline de referencia para predecir si un cliente suscribirá un depósito a término.
Es una **línea base**: tu trabajo es mejorarla.

Puntos clave:
- Todo el preprocesamiento va **dentro** del `Pipeline`: la plataforma le pasa al modelo las columnas crudas, sin `y`.
- El ranking ordena por **precision**, pero solo cuentan los envíos con **recall ≥ 0.30**.

## 1. Instalar las versiones de la plataforma

En Colab, ejecuta esta celda y luego **reinicia la sesión** (*Entorno de ejecución → Reiniciar sesión*). En tu computador usa `pip install -r requirements.txt`.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/estemen27/ml1-starter-kit/main/requirements.txt

In [ ]:
import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("scikit-learn", sklearn.__version__, "(debe ser 1.6.1)")

## 2. Cargar los datos

Usa los archivos que te entregó el profesor: `X_train.csv` (columnas crudas) y `y_train.csv` (columna `y`). En Colab súbelos con el ícono de carpeta del panel izquierdo.

In [ ]:
X = pd.read_csv("X_train.csv")
y = pd.read_csv("y_train.csv")["y"]

print(X.shape)
print(y.value_counts(normalize=True))
X.head()

## 3. Pipeline: preprocesamiento + modelo

`make_column_selector` elige las columnas por tipo, así el pipeline funciona con las columnas crudas tal como llegan.

In [ ]:
preprocesamiento = ColumnTransformer([
    ("numericas", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
     make_column_selector(dtype_include="number")),
    ("categoricas", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")),
     make_column_selector(dtype_include=object)),
])

modelo = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("clasificador", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

## 4. Evaluar con validación propia

Separa una parte de **tus** datos para estimar el desempeño. Mira precision **y** recall: con recall < 0.30 tu envío no entra al ranking.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
modelo.fit(X_tr, y_tr)
print(classification_report(y_val, modelo.predict(X_val), digits=4))

## 5. Entrenar con todos los datos y guardar

In [ ]:
modelo.fit(X, y)
# compress=3 reduce mucho el tamaño (máximo 50 MB); la plataforma lo carga igual
joblib.dump(modelo, "modelo.pkl", compress=3)

## 6. Validar antes de subir

Si termina en `RESULTADO: listo para subir`, sube `modelo.pkl` en la plataforma (**Mis envíos**).

In [ ]:
!wget -q -N https://raw.githubusercontent.com/estemen27/ml1-starter-kit/main/validar_modelo.py
!wget -q -N https://raw.githubusercontent.com/estemen27/ml1-starter-kit/main/requirements.txt
!python validar_modelo.py modelo.pkl --datos X_train.csv --y y_train.csv